In [1]:
import torch
import os
os.chdir('../../')

In [2]:
from scipy import linalg
import numpy as np
import os, torch
from tqdm import tqdm
from reports.util import load_config


@torch.no_grad()
def _trace_sqrtm_product(C1: torch.Tensor, C2: torch.Tensor) -> torch.Tensor:
    # Tr sqrtm(C1 @ C2) = Tr sqrt( C1^{1/2} C2 C1^{1/2} )
    s, U = torch.linalg.eigh(C1)                 # C1 = U diag(s) U^T
    s = s.clamp_min(0)
    C1h = (U * s.sqrt()) @ U.t()                 # C1^{1/2}
    M   = C1h @ C2 @ C1h
    w   = torch.linalg.eigvalsh((M + M.t()) * 0.5).clamp_min(0)
    return w.sqrt().sum()

@torch.no_grad()
def calc_fid_stats(mu1, sigma1, mu2, sigma2, eps: float = 1e-6) -> float:
    # 모두 float64 + 동일 device로 정렬
    C1 = torch.as_tensor(sigma1, dtype=torch.float64)
    device = C1.device
    C2 = torch.as_tensor(sigma2, dtype=torch.float64).to(device)
    m1 = torch.as_tensor(mu1,    dtype=torch.float64).to(device).flatten()
    m2 = torch.as_tensor(mu2,    dtype=torch.float64).to(device).flatten()

    D = m1.numel()
    I = torch.eye(D, dtype=torch.float64, device=device)

    # 대칭화 + 정칙화
    C1 = (C1 + C1.t()) * 0.5 + eps * I
    C2 = (C2 + C2.t()) * 0.5 + eps * I

    diff = m1 - m2
    tr_covmean = _trace_sqrtm_product(C1, C2)
    fid = diff.dot(diff) + torch.trace(C1) + torch.trace(C2) - 2.0 * tr_covmean
    return float(fid)

@torch.no_grad()
def calc_fid_pt_dir(pt_dir: str, mu, sigma, eps: float = 1e-6, num=100000, key="inception_feature") -> float:
    # pt_dir에서 'inception_feature'를 모아서 mu1, sigma1 추정 후 FID 계산
    X = []
    for f in tqdm(os.listdir(pt_dir)[:num]):
        if f.endswith(".pt"):
            v = torch.load(os.path.join(pt_dir, f), map_location="cpu").get(key)
            if v is not None:
                X.append(torch.as_tensor(v, dtype=torch.float64).flatten())
    if len(X) < 2:
        raise ValueError("need >=2 features")

    X   = torch.stack(X, 0)                 # [N, D]
    mu1 = X.mean(0)
    Xc  = X - mu1
    sigma1 = (Xc.t() @ Xc) / (X.shape[0] - 1)  # 불편추정

    return calc_fid_stats(mu1, sigma1, mu, sigma, eps=eps)
    #return calculate_frechet_distance(mu1, sigma1, mu, sigma, eps=eps)

def calculate_frechet_distance(mu1, sigma1, mu2, sigma2, eps=1e-6):
    """Numpy implementation of the Frechet Distance.
    The Frechet distance between two multivariate Gaussians X_1 ~ N(mu_1, C_1)
    and X_2 ~ N(mu_2, C_2) is
            d^2 = ||mu_1 - mu_2||^2 + Tr(C_1 + C_2 - 2*sqrt(C_1*C_2)).

    Stable version by Dougal J. Sutherland.

    Params:
    -- mu1   : Numpy array containing the activations of a layer of the
               inception net (like returned by the function 'get_predictions')
               for generated samples.
    -- mu2   : The sample mean over activations, precalculated on an
               representative data set.
    -- sigma1: The covariance matrix over activations for generated samples.
    -- sigma2: The covariance matrix over activations, precalculated on an
               representative data set.

    Returns:
    --   : The Frechet Distance.
    """

    mu1 = np.atleast_1d(mu1)
    mu2 = np.atleast_1d(mu2)

    sigma1 = np.atleast_2d(sigma1)
    sigma2 = np.atleast_2d(sigma2)

    assert mu1.shape == mu2.shape, \
        'Training and test mean vectors have different lengths'
    assert sigma1.shape == sigma2.shape, \
        'Training and test covariances have different dimensions'

    diff = mu1 - mu2

    # Product might be almost singular
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
    if not np.isfinite(covmean).all():
        msg = ('fid calculation produces singular product; '
               'adding %s to diagonal of cov estimates') % eps
        print(msg)
        offset = np.eye(sigma1.shape[0]) * eps
        covmean = linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))

    # Numerical error might give slight imaginary component
    if np.iscomplexobj(covmean):
        if not np.allclose(np.diagonal(covmean).imag, 0, atol=1e-3):
            m = np.max(np.abs(covmean.imag))
            raise ValueError('Imaginary component {}'.format(m))
        covmean = covmean.real

    tr_covmean = np.trace(covmean)

    return (diff.dot(diff) + np.trace(sigma1)
            + np.trace(sigma2) - 2 * tr_covmean)

def get_clip_score(pt_file, key):
    data = torch.load(pt_file)
    return float(data[key])

from pathlib import Path
import numpy as np
from tqdm import tqdm

def get_clip_scores(dir):
    scores = {}
    
    for pt_file in tqdm(Path(dir).rglob('*.pt')):
        data = torch.load(pt_file)
        for key in data.keys():
            if key.startswith('clip_score'):
                clip_score = get_clip_score(pt_file, key)
                if key in scores:
                    scores[key].append(clip_score)
                else:
                    scores[key] = [clip_score]
    for key in scores.keys():
        scores[key] = np.mean(scores[key])
    return scores
        

In [10]:
pt_dirs = [
        #'samplings/SANA/4.5/9/Euler/30000/euler_mjhq/euler_mjhq_0',
        #'samplings/SANA/4.5/8/Euler/30000/euler_mjhq/euler_mjhq_0',
        #'samplings/SANA/4.5/7/Euler/30000/euler_mjhq/euler_mjhq_0',
        #'samplings/SANA/4.5/6/Euler/30000/euler_mjhq/euler_mjhq_0',
        #'samplings/SANA/4.5/5/Euler/30000/euler_mjhq/euler_mjhq_0',
        #'samplings/SANA/4.5/4/Euler/30000/euler_mjhq/euler_mjhq_0',
        'samplings/SANA/4.5/3/Euler/30000/euler_mjhq/euler_mjhq_0',
        ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    data = torch.load('mjhq_fid/mjhq_30k_fid_stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    scores = get_clip_scores(pt_dir)
    print(pt_dir)
    print('FID :', fid)
    for key in scores.keys():
        print(key, f"{scores[key]:.4f}")

100%|██████████| 30001/30001 [00:29<00:00, 1031.27it/s]
30000it [01:25, 351.65it/s]

samplings/SANA/4.5/3/Euler/30000/euler_mjhq/euler_mjhq_0
FID : 44.366175604519015
clip_score_ViT-L/14 0.2469
clip_score_ViT-L/14@336px 0.2513
clip_score_RN101 0.4697


In [11]:
pt_dirs = [
        #'samplings/SANA/4.5/9/DPM-Solver/30000/dpm_mjhq/dpm_mjhq_0',
        #'samplings/SANA/4.5/8/DPM-Solver/30000/dpm_mjhq/dpm_mjhq_0',
        #'samplings/SANA/4.5/7/DPM-Solver/30000/dpm_mjhq/dpm_mjhq_0',
        #'samplings/SANA/4.5/6/DPM-Solver/30000/dpm_mjhq/dpm_mjhq_0',
        #'samplings/SANA/4.5/5/DPM-Solver/30000/dpm_mjhq/dpm_mjhq_0',
        #'samplings/SANA/4.5/4/DPM-Solver/30000/dpm_mjhq/dpm_mjhq_0',
        'samplings/SANA/4.5/3/DPM-Solver/30000/dpm_mjhq/dpm_mjhq_0',
        ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    data = torch.load('mjhq_fid/mjhq_30k_fid_stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    scores = get_clip_scores(pt_dir)
    print(pt_dir)
    print('FID :', fid)
    for key in scores.keys():
        print(key, f"{scores[key]:.4f}")

  0%|          | 0/30001 [00:00<?, ?it/s]

100%|██████████| 30001/30001 [00:21<00:00, 1376.61it/s]
30000it [01:25, 352.79it/s]

samplings/SANA/4.5/3/DPM-Solver/30000/dpm_mjhq/dpm_mjhq_0
FID : 43.378031107110814
clip_score_ViT-L/14 0.2467
clip_score_ViT-L/14@336px 0.2512
clip_score_RN101 0.4697


In [4]:
pt_dirs = [
        # 'samplings/SANA/4.5/9/BNS-Solver/30000/bns_mjhq_0/',
        # 'samplings/SANA/4.5/8/BNS-Solver/30000/bns_mjhq_0/',
        #'samplings/SANA/4.5/7/BNS-Solver/30000/bns_mjhq_0/',
        #'samplings/SANA/4.5/6/BNS-Solver/30000/bns_mjhq_0/',
        'samplings/SANA/4.5/5/BNS-Solver/30000/bns_mjhq_0/',
        #'samplings/SANA/4.5/4/BNS-Solver/30000/bns_mjhq_0/',
        #'samplings/SANA/4.5/3/BNS-Solver/30000/bns_mjhq_0/',
        ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    data = torch.load('mjhq_fid/mjhq_30k_fid_stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    scores = get_clip_scores(pt_dir)
    print(pt_dir)
    print('FID :', fid)
    for key in scores.keys():
        print(key, f"{scores[key]:.4f}")

100%|██████████| 30001/30001 [00:44<00:00, 674.83it/s]
30000it [02:06, 237.77it/s]

samplings/SANA/4.5/5/BNS-Solver/30000/bns_mjhq_0/
FID : 12.994838105490828
clip_score_ViT-B/16 0.3310
clip_score_ViT-L/14 0.2824
clip_score_ViT-L/14@336px 0.2890
clip_score_RN101 0.5033


In [11]:
pt_dirs = [
        #'samplings/SANA/4.5/9/DS-Solver_Flow/30000/ds_mjhq_0/',
        #'samplings/SANA/4.5/8/DS-Solver_Flow/30000/ds_mjhq_0/',
        'samplings/SANA/4.5/7/DS-Solver_Flow/30000/ds_mjhq_0/',
        'samplings/SANA/4.5/6/DS-Solver_Flow/30000/ds_mjhq_0/',
        'samplings/SANA/4.5/5/DS-Solver_Flow/30000/ds_mjhq_0/',
        'samplings/SANA/4.5/4/DS-Solver_Flow/30000/ds_mjhq_0/',
        'samplings/SANA/4.5/3/DS-Solver_Flow/30000/ds_mjhq_0/',
        ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    data = torch.load('mjhq_fid/mjhq_30k_fid_stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    scores = get_clip_scores(pt_dir)
    print(pt_dir)
    print('FID :', fid)
    for key in scores.keys():
        print(key, f"{scores[key]:.4f}")

  0%|          | 0/30001 [00:00<?, ?it/s]

100%|██████████| 30001/30001 [00:15<00:00, 1885.68it/s]
30000it [01:14, 402.40it/s]


samplings/SANA/4.5/7/DS-Solver_Flow/30000/ds_mjhq_0/
FID : 7.821580205466205
clip_score_ViT-B/16 0.3360
clip_score_ViT-L/14 0.2869
clip_score_ViT-L/14@336px 0.2940
clip_score_RN101 0.5063


100%|██████████| 30001/30001 [00:16<00:00, 1808.10it/s]
30000it [01:14, 400.55it/s]


samplings/SANA/4.5/6/DS-Solver_Flow/30000/ds_mjhq_0/
FID : 9.539683467594102
clip_score_ViT-B/16 0.3342
clip_score_ViT-L/14 0.2850
clip_score_ViT-L/14@336px 0.2918
clip_score_RN101 0.5047


...

samplings/SANA/4.5/5/DS-Solver_Flow/30000/ds_mjhq_0/
FID : 13.162514709289724
clip_score_ViT-B/16 0.3312
clip_score_ViT-L/14 0.2815
clip_score_ViT-L/14@336px 0.2880
clip_score_RN101 0.5014


100%|██████████| 30001/30001 [00:15<00:00, 1879.92it/s]
30000it [01:14, 403.90it/s]


samplings/SANA/4.5/4/DS-Solver_Flow/30000/ds_mjhq_0/
FID : 23.437465123994173
clip_score_ViT-B/16 0.3215
clip_score_ViT-L/14 0.2708
clip_score_ViT-L/14@336px 0.2765
clip_score_RN101 0.4913


100%|██████████| 30001/30001 [00:16<00:00, 1796.16it/s]
30000it [01:15, 398.02it/s]

samplings/SANA/4.5/3/DS-Solver_Flow/30000/ds_mjhq_0/
FID : 45.845596828786654
clip_score_ViT-B/16 0.2991
clip_score_ViT-L/14 0.2467
clip_score_ViT-L/14@336px 0.2509
clip_score_RN101 0.4696


In [3]:
pt_dirs = [
        #'samplings/SANA/4.5/9/Dual-Solver/30000/rn_mjhq_0/',
        # 'samplings/SANA/4.5/8/Dual-Solver/30000/rn_mjhq_0/',
        # 'samplings/SANA/4.5/7/Dual-Solver/30000/rn_mjhq_0/',
        # 'samplings/SANA/4.5/6/Dual-Solver/30000/rn_mjhq_0/',
        'samplings/SANA/4.5/5/Dual-Solver/30000/rn_mjhq_0/',
        #'samplings/SANA/4.5/4/Dual-Solver/30000/rn_mjhq_0/',
        #'samplings/SANA/4.5/3/Dual-Solver/30000/rn_mjhq_0/',
        ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    data = torch.load('mjhq_fid/mjhq_30k_fid_stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    scores = get_clip_scores(pt_dir)
    print(pt_dir)
    print('FID :', fid)
    for key in scores.keys():
        print(key, f"{scores[key]:.4f}")

100%|██████████| 30001/30001 [00:44<00:00, 675.11it/s]
30000it [02:09, 232.36it/s]

samplings/SANA/4.5/5/Dual-Solver/30000/rn_mjhq_0/
FID : 11.503636116502491
clip_score_ViT-B/16 0.3316
clip_score_ViT-L/14 0.2830
clip_score_ViT-L/14@336px 0.2891
clip_score_RN101 0.5025


In [16]:
pt_dirs = [
        'samplings/SANA/4.5/9/Dual-Solver/30000/rn_trained_mjhq_0/',
        ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    data = torch.load('mjhq_fid/mjhq_30k_fid_stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    scores = get_clip_scores(pt_dir)
    print(pt_dir)
    print('FID :', fid)
    for key in scores.keys():
        print(key, f"{scores[key]:.4f}")

100%|██████████| 30001/30001 [00:17<00:00, 1737.23it/s]
30000it [01:15, 399.62it/s]

samplings/SANA/4.5/9/Dual-Solver/30000/rn_trained_mjhq_0/
FID : 7.738092162503733
clip_score_ViT-B/16 0.3356
clip_score_ViT-L/14 0.2880
clip_score_ViT-L/14@336px 0.2931
clip_score_RN101 0.5074


In [11]:
pt_dirs = [
        #'samplings/SANA/4.5/9/Dual-Solver/30000/traj_mjhq_0/',
        #'samplings/SANA/4.5/8/Dual-Solver/30000/traj_mjhq_0/',
        # 'samplings/SANA/4.5/7/Dual-Solver/30000/traj_mjhq_0/',
        # 'samplings/SANA/4.5/6/Dual-Solver/30000/traj_mjhq_0/',
        # 'samplings/SANA/4.5/5/Dual-Solver/30000/traj_mjhq_0/',
        'samplings/SANA/4.5/4/Dual-Solver/30000/traj_mjhq_0/',
        #'samplings/SANA/4.5/3/Dual-Solver/30000/traj_mjhq_0/',
        ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    data = torch.load('mjhq_fid/mjhq_30k_fid_stats.pt')
    print(pt_dir)

    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print('FID :', fid)

    scores = get_clip_scores(pt_dir)
    for key in scores.keys():
        print(key, f"{scores[key]:.4f}")

samplings/SANA/4.5/4/Dual-Solver/30000/traj_mjhq_0/


  0%|          | 0/30001 [00:00<?, ?it/s]

100%|██████████| 30001/30001 [00:16<00:00, 1851.45it/s]


FID : 14.699475578229908


30000it [01:15, 395.89it/s]

clip_score_ViT-B/16 0.3286
clip_score_ViT-L/14 0.2791
clip_score_ViT-L/14@336px 0.2858
clip_score_RN101 0.5010
